# 10 — Protocolos adversariais do paper e de `advex.py`

Reproduzimos separadamente três protocolos de Wang et al. (arXiv:2211.00593):

1. **paper IO single** (Fig. 8): uma das cinco cláusulas naturais contém IO uma vez;
2. **paper S control** (Fig. 8): a mesma família de cláusulas contém S uma vez;
3. **advex double IO**: extensão executável de `Easy-Transformer/easy_transformer/advex.py` no commit `ea15315dd24481e9e2ac5c3ef335d82907a1dc34`, que sorteia um dos 25 pares ordenados e insere IO duas vezes.

Os três braços compartilham a mesma população base e cada posição adversarial é recalculada após a inserção. Somente o braço **paper IO single** é comparado numericamente aos alvos adversariais 1.23/23.4% da Fig. 8.

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
import json
from pathlib import Path
import subprocess

import pandas as pd
import plotly.express as px
import torch

from ioi_replication.paper import (
    DOUBLE_ADVERSARIAL_TEMPLATES,
    EASY_TRANSFORMER_ADVEX_URL,
    EASY_TRANSFORMER_SOURCE_COMMIT,
    PAPER_FIGURE_8_TARGETS,
    build_adversarial_dataset,
    io_probability,
    logit_difference,
    summarize_adversarial_values,
)
from ioi_replication.runtime import load_gpt2_small, select_device

BASE_SEED = 510
PAPER_ATTACK_SEED = 511
ADVEX_ATTACK_SEED = 512
N = 500
BATCH_SIZE = 16
MODEL_NAME = "gpt2"
ALLOWED_TEMPLATE_IDS = tuple(range(7))
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
device = select_device()
print(
    f"Dispositivo: {device}; base seed: {BASE_SEED}; N: {N}; "
    f"batch: {BATCH_SIZE}"
)

## População, fonte e correção de posição

A população histórica `mixed` de `IOIDataset(prompt_type="mixed", nb_templates=15)` sofre um off-by-one: usa `BABA_TEMPLATES[: 15 // 2] + ABBA_TEMPLATES[: 15 // 2]`. Portanto, estes braços usam exatamente os IDs 0..6 de BABA e 0..6 de ABBA (14 templates). Nomes são filtrados para tokens únicos no GPT-2; isso é uma diferença explícita da geração histórica, que assumia a lista de nomes sem validar a tokenização.

Há uma inconsistência BOS/`punct` no `advex.py` histórico. `punct_idx` vem de `IOIDataset` sem BOS, mas `show_tokens` chama `model.to_tokens`, que historicamente adiciona BOS por padrão; o índice e a lista de tokens ficam em sistemas de coordenadas diferentes. Aqui localizamos a pontuação intermediária no texto, inserimos a cláusula e retokenizamos tudo sem BOS. Recalcular IO/S1/S2/END e as ocorrências inseridas é uma **correção intencional**, não uma tentativa de reproduzir esse deslocamento acidental.

A Fig. 8 reporta, para uma ocorrência adicional de IO, IO−S = 1.23, P(IO) = 0.36 e S>IO = 23.4%. O controle com uma ocorrência adicional de S também aparece na Fig. 8, mas a comparação quantitativa principal deste notebook é restrita ao protocolo paper IO single. O braço `advex.py` de duas cláusulas é uma extensão histórica e não recebe esses targets.

In [ ]:
model = load_gpt2_small(device)
common = dict(
    n=N,
    base_seed=BASE_SEED,
    pattern="mixed",
    allowed_template_ids=ALLOWED_TEMPLATE_IDS,
)
paper_io_single = build_adversarial_dataset(
    model.tokenizer,
    attack_seed=PAPER_ATTACK_SEED,
    role="IO",
    clause_count="single",
    **common,
)
paper_s_control = build_adversarial_dataset(
    model.tokenizer,
    attack_seed=PAPER_ATTACK_SEED,
    role="S",
    clause_count="single",
    **common,
)
advex_double_io = build_adversarial_dataset(
    model.tokenizer,
    attack_seed=ADVEX_ATTACK_SEED,
    role="IO",
    clause_count="double",
    **common,
)
datasets = {
    "paper_io_single": paper_io_single,
    "paper_s_control": paper_s_control,
    "advex_double_io": advex_double_io,
}

assert len(DOUBLE_ADVERSARIAL_TEMPLATES) == 25
assert all(dataset.prepend_bos is False for dataset in datasets.values())
assert model.tokenizer.padding_side == "right"
assert all(
    dataset.base_prompts == paper_io_single.base_prompts
    for dataset in datasets.values()
)
assert all(
    set(dataset.template_ids.tolist()) == set(ALLOWED_TEMPLATE_IDS)
    for dataset in datasets.values()
)
for name, dataset in datasets.items():
    assert torch.equal(
        dataset.adversarial_positions["END"],
        dataset.adversarial_attention_mask.sum(dim=1) - 2,
    )
    print(f"\n{name}")
    print("baseline:", dataset.base_prompts[0])
    print("ataque:", dataset.adversarial_prompts[0])
    length = int(dataset.adversarial_attention_mask[0].sum())
    print(
        "tokens:",
        model.tokenizer.convert_ids_to_tokens(
            dataset.adversarial_tokens[0, :length].tolist()
        ),
    )
    print(
        "posições:",
        {key: int(value[0]) for key, value in dataset.adversarial_positions.items()},
        "inseridas:",
        dataset.inserted_positions[0].tolist(),
    )

In [ ]:
def evaluate_tokens(tokens: torch.Tensor, end_positions: torch.Tensor):
    differences = []
    probabilities = []
    with torch.inference_mode():
        for start in range(0, N, BATCH_SIZE):
            stop = min(start + BATCH_SIZE, N)
            logits = model(tokens[start:stop].to(device))
            differences.append(
                logit_difference(
                    logits,
                    paper_io_single.io_token_ids[start:stop].to(device),
                    paper_io_single.s_token_ids[start:stop].to(device),
                    end_positions[start:stop].to(device),
                    reduction="none",
                ).cpu()
            )
            probabilities.append(
                io_probability(
                    logits,
                    paper_io_single.io_token_ids[start:stop].to(device),
                    end_positions[start:stop].to(device),
                    reduction="none",
                ).cpu()
            )
            del logits
    return torch.cat(differences), torch.cat(probabilities)


# O baseline é executado uma única vez e reutilizado nos três deltas pareados.
baseline_ld, baseline_io_probability = evaluate_tokens(
    paper_io_single.base_tokens,
    paper_io_single.base_positions["END"],
)
arm_values = {}
paired_metrics = {}
for name, dataset in datasets.items():
    arm_ld, arm_probability = evaluate_tokens(
        dataset.adversarial_tokens,
        dataset.adversarial_positions["END"],
    )
    arm_values[name] = {
        "logit_difference": arm_ld,
        "io_probability": arm_probability,
    }
    paired_metrics[name] = summarize_adversarial_values(
        baseline_logit_difference=baseline_ld,
        adversarial_logit_difference=arm_ld,
        baseline_io_probability=baseline_io_probability,
        adversarial_io_probability=arm_probability,
    )
    print(
        name,
        "IO−S:", paired_metrics[name].logit_difference.adversarial.mean,
        "S>IO:", paired_metrics[name].s_wins_error_rate.adversarial.mean,
        "P(IO):", paired_metrics[name].io_probability.adversarial.mean,
    )

In [ ]:
def estimate_dict(estimate):
    return {
        "mean": estimate.mean,
        "standard_error": estimate.standard_error,
        "ci95": list(estimate.ci95),
        "ci_method": estimate.ci_method,
    }


def arm_metrics_dict(metrics, arm):
    return {
        "logit_difference_io_minus_s": estimate_dict(
            getattr(metrics.logit_difference, arm)
        ),
        "s_wins_error_rate": estimate_dict(
            getattr(metrics.s_wins_error_rate, arm)
        ),
        "io_probability": estimate_dict(
            getattr(metrics.io_probability, arm)
        ),
    }


baseline_metrics_json = arm_metrics_dict(
    paired_metrics["paper_io_single"], "baseline"
)
protocols_json = {
    "baseline": {"attack": None, "metrics": baseline_metrics_json},
}
for name, metrics in paired_metrics.items():
    dataset = datasets[name]
    protocols_json[name] = {
        "attack": {
            "role": dataset.attack_role,
            "clause_count": dataset.clause_count,
        },
        "paper_figure_8_protocol": name != "advex_double_io",
        "metrics": arm_metrics_dict(metrics, "adversarial"),
        "delta_from_baseline": arm_metrics_dict(metrics, "delta"),
    }

io_targets = PAPER_FIGURE_8_TARGETS["paper_io_single"]
comparison_to_paper = {
    "paper_io_single": {
        "logit_difference_io_minus_s": (
            protocols_json["paper_io_single"]["metrics"]
            ["logit_difference_io_minus_s"]["mean"]
            - io_targets["logit_difference"]
        ),
        "s_wins_error_rate": (
            protocols_json["paper_io_single"]["metrics"]
            ["s_wins_error_rate"]["mean"]
            - io_targets["s_wins_error_rate"]
        ),
        "io_probability": (
            protocols_json["paper_io_single"]["metrics"]
            ["io_probability"]["mean"]
            - io_targets["io_probability"]
        ),
    }
}
comparison = pd.DataFrame(
    [
        {
            "protocolo": name,
            "IO−S": protocol["metrics"]["logit_difference_io_minus_s"]["mean"],
            "S>IO": protocol["metrics"]["s_wins_error_rate"]["mean"],
            "P(IO)": protocol["metrics"]["io_probability"]["mean"],
            "target aplicável": name == "paper_io_single",
        }
        for name, protocol in protocols_json.items()
    ]
)
comparison

In [ ]:
distribution = pd.DataFrame(
    {
        "baseline": baseline_ld.numpy(),
        **{
            name: values["logit_difference"].numpy()
            for name, values in arm_values.items()
        },
    }
).melt(var_name="protocolo", value_name="IO−S")
fig_distribution = px.histogram(
    distribution,
    x="IO−S",
    color="protocolo",
    barmode="overlay",
    opacity=0.45,
    marginal="box",
    title="Distribuições de IO−S por protocolo",
)
fig_distribution.add_vline(x=0, line_dash="dash", line_color="black")
fig_distribution.show()

delta_rows = []
for name, values in arm_values.items():
    for delta_ld, delta_probability in zip(
        values["logit_difference"] - baseline_ld,
        values["io_probability"] - baseline_io_probability,
    ):
        delta_rows.append(
            {
                "protocolo": name,
                "delta IO−S": float(delta_ld),
                "delta P(IO)": float(delta_probability),
            }
        )
fig_delta = px.scatter(
    pd.DataFrame(delta_rows),
    x="delta IO−S",
    y="delta P(IO)",
    color="protocolo",
    marginal_x="histogram",
    marginal_y="histogram",
    title="Deltas pareados contra o baseline único",
)
fig_delta.add_vline(x=0, line_dash="dash", line_color="black")
fig_delta.add_hline(y=0, line_dash="dash", line_color="black")
fig_delta.show()

## Mecanismo previsto e limite causal

No protocolo paper IO single, a nova ocorrência do IO pode acionar a heurística dos heads **Duplicate Token** como se IO fosse o nome repetido a ser evitado. O paper observa que heads de **S-Inhibition** passam a atender também à segunda ocorrência de IO, reduzindo o encaminhamento do IO pelos Name Movers. O controle paper S single testa se o efeito vem apenas de acrescentar uma sentença: nele, reforçar a repetição correta de S deve preservar ou melhorar a preferência por IO. O protocolo `advex.py` double IO intensifica a perturbação, mas não é o protocolo associado aos targets da Fig. 8.

Esse resultado comportamental sozinho **não prova o circuito**. Ele confirma uma previsão, mas outras rotas do modelo e mudanças distribucionais podem produzir o mesmo padrão. Como o próprio paper ressalta, o comportamento de S-Inhibition nesses exemplos está além da análise causal original.

In [ ]:
source_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=PROJECT_ROOT,
    text=True,
).strip()
git_dirty = bool(
    subprocess.check_output(
        ["git", "status", "--porcelain"],
        cwd=PROJECT_ROOT,
        text=True,
    ).strip()
)
assert not git_dirty, "execute from the committed source tree for provenance"

source_paths = {
    "paper/adversarial.py": "src/ioi_replication/paper/adversarial.py",
    "paper/dataset.py": "src/ioi_replication/paper/dataset.py",
    "paper/metrics.py": "src/ioi_replication/paper/metrics.py",
    "runtime.py": "src/ioi_replication/runtime.py",
    "notebook": "notebooks/10_adversarial_examples.ipynb",
}


def git_blob(relative_path):
    return subprocess.check_output(
        ["git", "rev-parse", f"{source_commit}:{relative_path}"],
        cwd=PROJECT_ROOT,
        text=True,
    ).strip()


source_blobs = {
    name: git_blob(relative_path)
    for name, relative_path in source_paths.items()
}
model_dtype = str(next(model.parameters()).dtype).removeprefix("torch.")
gpu = torch.cuda.get_device_name(device) if device.type == "cuda" else None

result = {
    "paper": "https://arxiv.org/abs/2211.00593",
    "config": {
        "base_seed": BASE_SEED,
        "attack_seeds": {
            "paper_io_single": PAPER_ATTACK_SEED,
            "paper_s_control": PAPER_ATTACK_SEED,
            "advex_double_io": ADVEX_ATTACK_SEED,
        },
        "n": N,
        "batch_size": BATCH_SIZE,
        "model_name": MODEL_NAME,
        "device": str(device),
        "gpu": gpu,
        "dtype": model_dtype,
        "dataset_sha256": {
            name: dataset.dataset_sha256
            for name, dataset in datasets.items()
        },
        "allowed_template_ids": list(ALLOWED_TEMPLATE_IDS),
        "mixed_population": {
            "ABBA": list(ALLOWED_TEMPLATE_IDS),
            "BABA": list(ALLOWED_TEMPLATE_IDS),
        },
        "prepend_bos": paper_io_single.prepend_bos,
        "padding_side": model.tokenizer.padding_side,
    },
    "protocols": protocols_json,
    "paper_figure_8_reference": PAPER_FIGURE_8_TARGETS,
    "comparison_to_paper": comparison_to_paper,
    "methodology": {
        "comparison_scope": (
            "Figure 8 targets are compared only with paper_io_single; "
            "advex_double_io is reported as a separate historical extension."
        ),
        "differences": [
            "Names are filtered to GPT-2 single-token names; the historical generator did not validate this explicitly.",
            "Base and attack-template RNGs use independent recorded seeds.",
            "TransformerLens 4.3 compatibility mode replaces EasyTransformer.",
            "Individual error-rate intervals are Wilson; paired deltas use normal 1.96×SE intervals over within-example differences.",
        ],
        "advex_source_inconsistency": (
            "advex.py obtains punct_idx from an IOIDataset without BOS, while "
            "show_tokens calls model.to_tokens with historical BOS-prefixed "
            "behavior, mixing token coordinate systems."
        ),
        "position_policy": (
            "Text insertion followed by complete no-BOS retokenization is an "
            "intentional correction; no shifted base position is reused."
        ),
        "causal_limit": (
            "Behavior alone does not prove mediation by Duplicate Token and "
            "S-Inhibition heads."
        ),
    },
    "traceability": {
        "source_commit": source_commit,
        "git_dirty": git_dirty,
        "historical_source_commit": EASY_TRANSFORMER_SOURCE_COMMIT,
        "historical_source_url": EASY_TRANSFORMER_ADVEX_URL,
        "hash_method": (
            "git rev-parse <source_commit>:<path> (normalized Git blob IDs)"
        ),
        "source_blobs": source_blobs,
        "timestamp_utc": datetime.now(timezone.utc).isoformat().replace(
            "+00:00", "Z"
        ),
        "versions": {
            "torch": torch.__version__,
            "transformer_lens": version("transformer-lens"),
            "pandas": version("pandas"),
            "plotly": version("plotly"),
        },
    },
}
output_path = PROJECT_ROOT / "results/paper/10_adversarial_examples.json"
output_path.write_text(
    json.dumps(result, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print("Resultado salvo em", output_path.relative_to(PROJECT_ROOT))
result